# Exercise 01

Points: 10

In the file 'AgentStateTable-1.tsv.zip', we have provided a data sample generated using a [Human Mobility Simulation](https://github.com/azufle/pol/tree/main?tab=readme-ov-file) [1].

In this notebook, we will explore the synthetic trajectories created by the agents in the simulation.




**Note:** The data sample was generated with OpenStreetMap data. The data and map images are provided under [OpenStreetMap, Open Database License (ODbL)](https://www.openstreetmap.org/copyright).

**References:**

[1] [Joon-Seok Kim, Hyunjee Jin, Hamdi Kavak, Ovi Chris Rouly, Andrew Crooks, Dieter Pfoser, Carola Wenk and Andreas Züfle, Location-Based Social Network Data Generation Based on Patterns of Life, IEEE International Conference on Mobile Data Management.](https://github.com/azufle/pol/tree/main?tab=readme-ov-file)

[2] Map images: ©OpenStreetMap contributors, ODbL

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import random
from shapely import wkt
import folium
from pyproj import Transformer

In [24]:
##
# We read the dataset using pandas, a helpful library for analyzing, cleaning, exploring, and manipulating data.
##
_sim_data = pd.read_csv('AgentStateTable-1.tsv.zip', sep='\t') # ! Enter path to 'AgentStateTable-1.tsv.zip' here
print(_sim_data.shape)


# Convert 'location' column with WKT strings representing geometries, to shapely geometry objects
_sim_data['location'] = _sim_data['location'].apply(wkt.loads)


# Convert the dataset to a geopandas dataframe. Geopandas is another library for working with geopatial datasets.
sim_data = gpd.GeoDataFrame(_sim_data, geometry='location', crs='EPSG:26916')

# Convert data from CRS 'EPSG:26916' to 'EPSG:4326' (you'll learn about this later in the course)
sim_data = sim_data.to_crs("EPSG:4326")
sim_data.head()

C:\Users\ramy0\AppData\Local\Temp\ipykernel_7820\3397212720.py:4: DtypeWarning: Columns (21,24) have mixed types. Specify dtype option on import or set low_memory=False.
  _sim_data = pd.read_csv('AgentStateTable-1.tsv.zip', sep='\t') # ! Enter path to 'AgentStateTable-1.tsv.zip' here


(1633174, 33)


,step,simulationTime,location,agentId,neighborhoodId,age,currentMode,currentUnit,family:classroom,lifeStatus,...,loveNeed:socialHappiness,visitReason,todaysPlan:day,todaysPlan:wakeUpTime,todaysPlan:leaveTimeForWork,todaysPlan:leaveTimeFromWork,todaysPlan:workDay,todaysPlan:beenAtWork,todaysPlan:cameBackFromWork,Unnamed: 32
0,0,2019-07-01T00:00:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,-0.001,NaN,2019-07-01,07:02:00.000,07:12:00.000,15:27:00.000,True,False,False,NaN
1,0,2019-07-01T00:00:00.000,POINT (7.09236 50.73909),1,0,21.083333,AtHome,2.0,0.0,Alive,...,-0.001,NaN,2019-07-01,07:34:00.000,07:44:00.000,15:59:00.000,True,False,False,NaN
2,0,2019-07-01T00:00:00.000,POINT (7.09155 50.73937),2,0,34.083333,AtHome,0.0,0.0,Alive,...,-0.001,NaN,2019-07-01,08:16:00.000,08:26:00.000,16:36:00.000,True,False,False,NaN
3,0,2019-07-01T00:00:00.000,POINT (7.09694 50.74011),3,0,27.083333,AtHome,0.0,0.0,Alive,...,-0.001,NaN,2019-07-01,06:32:00.000,06:42:00.000,15:02:00.000,True,False,False,NaN
4,0,2019-07-01T00:00:00.000,POINT (7.08876 50.74119),4,0,53.083333,AtHome,16.0,0.0,Alive,...,-0.001,NaN,2019-07-01,07:13:00.000,07:23:00.000,15:33:00.000,True,False,False,NaN


In [26]:
# Extract latitude and longitude
sim_data['latitude'] = sim_data['location'].apply(lambda point: point.y)
sim_data['longitude'] = sim_data['location'].apply(lambda point: point.x)


In [4]:
# Checking the rows and columns in the dataset
sim_data.shape, sim_data.columns

((1633174, 35),
 Index(['step', 'simulationTime', 'location', 'agentId', 'neighborhoodId',
        'age', 'currentMode', 'currentUnit', 'family:classroom', 'lifeStatus',
        'foodNeed:fullness', 'foodNeed:status', 'foodNeed:numberOfMealsTaken',
        'sleepNeed:status', 'shelterNeed:currentShelter',
        'financialSafetyNeed:availableBalance', 'financialSafetyNeed:job:id',
        'financialSafetyNeed:status', 'financialSafetyNeed:dailyFoodBudget',
        'financialSafetyNeed:weeklyExtraBudget', 'loveNeed:meetingId',
        'loveNeed:status', 'loveNeed:socialStatus', 'loveNeed:socialHappiness',
        'visitReason', 'todaysPlan:day', 'todaysPlan:wakeUpTime',
        'todaysPlan:leaveTimeForWork', 'todaysPlan:leaveTimeFromWork',
        'todaysPlan:workDay', 'todaysPlan:beenAtWork',
        'todaysPlan:cameBackFromWork', 'Unnamed: 32', 'latitude', 'longitude'],
       dtype='object'))

In [5]:
# Let's see how many simulated agents are present in the data
sim_data['agentId'].value_counts() # -- 1000 agents, with each agent taking around 1634 steps

agentId
125    1634
170    1634
15     1634
171    1634
0      1634
       ... 
995    1633
996    1633
997    1633
998    1633
874    1633
Name: count, Length: 1000, dtype: int64

In [8]:
# In which city do the simulated agents live? Let's check this by plotting the first 100 rows on a map.
sim_data_sliced = sim_data.head(100)

# Create a map centered at the mean latitude and longitude coordinates
m = folium.Map(location=[sim_data_sliced['latitude'].mean(), sim_data_sliced['longitude'].mean()], zoom_start=13)

# Iterate over rows and plot the locations. We put the location data in a tooltip, which see when you hover on a location
for index, row in sim_data_sliced.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color='blue',
        fill=True,
        fill_color='blue',
        tooltip=f"Location: {row['location']}<br>Timestamp: {row['simulationTime']}<br>Agent ID: {row['agentId']}"
    ).add_to(m)
m


In [10]:
# How old are the agents?
sim_data[['agentId','age']].value_counts()

agentId  age      
125      52.083333    1634
170      41.083333    1634
15       43.083333    1634
171      33.083333    1634
0        52.083333    1634
                      ... 
995      25.083333    1633
996      53.083333    1633
997      47.083333    1633
998      23.083333    1633
874      52.083333    1633
Name: count, Length: 1000, dtype: int64

In [12]:
# Which days are present in the simulated data?
sim_data['todaysPlan:day'].value_counts()

todaysPlan:day
2019-07-01    288031
2019-07-03    287999
2019-07-04    287999
2019-07-02    287998
2019-07-05    287997
2019-07-06    193150
Name: count, dtype: int64

In [14]:
# Possible modes of an agent
sim_data['currentMode'].value_counts()

currentMode
AtHome          860958
AtWork          472195
AtRecreation    204339
Transport        66356
AtRestaurant     29326
Name: count, dtype: int64

In [15]:
# Common reasons for the agents to visit locations
sim_data['visitReason'].value_counts()

visitReason
Home_ComingBackFromPub         9369
Restaurant_WantToEatOutside    8131
ComingBackFromRestaurant       6989
Bar_Proximity                  5065
Workplace_Work                 4900
Home_ComingBackFromWork        4785
Bar_SimilarIncome              3042
Bar_SimilarInterest            2958
Home_WantToSleep               1777
Bar_SimilarAge                  334
Name: count, dtype: int64

In [16]:
# Let's see what agent 0 has been doing in the simulation
agent0_data = sim_data[sim_data['agentId'] == 0]
agent0_data

,step,simulationTime,location,agentId,neighborhoodId,age,currentMode,currentUnit,family:classroom,lifeStatus,...,todaysPlan:day,todaysPlan:wakeUpTime,todaysPlan:leaveTimeForWork,todaysPlan:leaveTimeFromWork,todaysPlan:workDay,todaysPlan:beenAtWork,todaysPlan:cameBackFromWork,Unnamed: 32,latitude,longitude
0,0,2019-07-01T00:00:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:27:00.000,True,False,False,NaN,50.745692,7.083086
1000,1,2019-07-01T00:05:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:27:00.000,True,False,False,NaN,50.745692,7.083086
2000,2,2019-07-01T00:10:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:27:00.000,True,False,False,NaN,50.745692,7.083086
3000,3,2019-07-01T00:15:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:27:00.000,True,False,False,NaN,50.745692,7.083086
4000,4,2019-07-01T00:20:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:27:00.000,True,False,False,NaN,50.745692,7.083086
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1629000,1629,2019-07-06T15:45:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-06,07:02:00.000,NaN,NaN,False,False,False,NaN,50.745692,7.083086
1630000,1630,2019-07-06T15:50:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-06,07:02:00.000,NaN,NaN,False,False,False,NaN,50.745692,7.083086
1631000,1631,2019-07-06T15:55:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-06,07:02:00.000,NaN,NaN,False,False,False,NaN,50.745692,7.083086
1632000,1632,2019-07-06T16:00:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-06,07:02:00.000,NaN,NaN,False,False,False,NaN,50.745692,7.083086


In [17]:
agent0_data['todaysPlan:day'].value_counts()

todaysPlan:day
2019-07-01    288
2019-07-02    288
2019-07-03    288
2019-07-04    288
2019-07-05    288
2019-07-06    194
Name: count, dtype: int64

In [18]:
##
# Function to plot trajectories of an agent
##

DATE2COLOR = {
    '2019-07-01': '#1f77b4',
    '2019-07-02': '#ff7f0e',
    '2019-07-03': '#2ca02c',
    '2019-07-04': '#d62728',
    '2019-07-05': '#9467bd',
    '2019-07-06': '#8c564b',
}

TOOLTIP_COLUMNS = ['step', 'simulationTime', 'location', 'agentId',
        'age', 'currentMode', 'lifeStatus',
        'foodNeed:fullness', 'foodNeed:status', 'foodNeed:numberOfMealsTaken',
        'sleepNeed:status', 'shelterNeed:currentShelter',
        'financialSafetyNeed:availableBalance', 'financialSafetyNeed:job:id',
        'financialSafetyNeed:status', 'financialSafetyNeed:dailyFoodBudget',
        'financialSafetyNeed:weeklyExtraBudget', 'loveNeed:meetingId',
        'loveNeed:status', 'loveNeed:socialStatus', 'loveNeed:socialHappiness',
        'visitReason', 'todaysPlan:day', 'todaysPlan:wakeUpTime',
        'todaysPlan:leaveTimeForWork', 'todaysPlan:leaveTimeFromWork',
        'todaysPlan:workDay', 'todaysPlan:beenAtWork',
        'todaysPlan:cameBackFromWork'
  ]


def plot_agent_trajectories(agent_data):
  '''
  Generates a Folium map visualizing the agent's trajectory.

  Args:
    agent_data: A pandas DataFrame containing the agent's data.

  Returns:
    A Folium map object with the agent's trajectories plotted.
  '''


  # Calculate average longitude and latitude
  avg_latitude = agent_data['latitude'].mean()
  avg_longitude = agent_data['longitude'].mean()

  # Initialize the map centered around the average location
  m = folium.Map(location=[avg_latitude, avg_longitude], zoom_start=15)

  # Sort agent's data by simulationTime
  agent_data = agent_data.sort_values(by='simulationTime')

  # Group agent's data by date and plot a trajectory for each date
  agent_data_grouped = agent_data.groupby('todaysPlan:day')

  for date, data in agent_data_grouped:
    prev_location = None
    for index, row in data.iterrows():
      current_location = (row['latitude'], row['longitude'])

      # All column names and values in tooltip
      tooltip_text = ""
      tooltip_text = "".join([f"{colname}: {row[colname]}<br>" for colname in TOOLTIP_COLUMNS if not pd.isna(row[colname])])

      if prev_location is None:
        # Different icon for start location
        folium.Marker(
            icon=folium.Icon(color='red'),
            location=current_location,
            radius=5,
            color='red',
            fill=True,
            fill_color='blue',
            tooltip=tooltip_text
        ).add_to(m)

      folium.CircleMarker(
          location=current_location,
          radius=5,
          color='blue',
          fill=True,
          fill_color='blue',
          tooltip=tooltip_text
      ).add_to(m)

      if prev_location:
        folium.PolyLine([prev_location, current_location], color= DATE2COLOR[date]).add_to(m)

      prev_location = current_location

  return m

In [19]:
# Taking a look at what agent0 was doing '2019-07-01'
agent0_data[agent0_data['todaysPlan:day'] == '2019-07-01']

,step,simulationTime,location,agentId,neighborhoodId,age,currentMode,currentUnit,family:classroom,lifeStatus,...,todaysPlan:day,todaysPlan:wakeUpTime,todaysPlan:leaveTimeForWork,todaysPlan:leaveTimeFromWork,todaysPlan:workDay,todaysPlan:beenAtWork,todaysPlan:cameBackFromWork,Unnamed: 32,latitude,longitude
0,0,2019-07-01T00:00:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:27:00.000,True,False,False,NaN,50.745692,7.083086
1000,1,2019-07-01T00:05:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:27:00.000,True,False,False,NaN,50.745692,7.083086
2000,2,2019-07-01T00:10:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:27:00.000,True,False,False,NaN,50.745692,7.083086
3000,3,2019-07-01T00:15:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:27:00.000,True,False,False,NaN,50.745692,7.083086
4000,4,2019-07-01T00:20:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:27:00.000,True,False,False,NaN,50.745692,7.083086
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283000,283,2019-07-01T23:35:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:57:00.000,True,True,True,NaN,50.745692,7.083086
284000,284,2019-07-01T23:40:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:57:00.000,True,True,True,NaN,50.745692,7.083086
285000,285,2019-07-01T23:45:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:57:00.000,True,True,True,NaN,50.745692,7.083086
286000,286,2019-07-01T23:50:00.000,POINT (7.08309 50.74569),0,0,52.083333,AtHome,1.0,0.0,Alive,...,2019-07-01,07:02:00.000,07:12:00.000,15:57:00.000,True,True,True,NaN,50.745692,7.083086


In [20]:
# Plotting agent0's trajectory on '2019-07-01'
plot_agent_trajectories(agent0_data[agent0_data['todaysPlan:day'] == '2019-07-01'])

# Task

**Question**: Now you will be assigned an agent whose behavior needs to be analysed. Run the below code cell to generate an 'agentId' (MY_AGENT_ID) and examine the following about this agent:

- How old is the agent? What time does it wake up every morning and what time does it leave for work? -- 3 points

- Which is the most frequently visited location of this agent and why does it visit this location? -- 3 points

- Plot the trajectory of this agent on '2019-07-02' and describe its behavior on the day based on the available data. You can use the plot_agent_trajectories function above for this task or create your own function. -- 4 points






In [68]:
# Run this cell, to generate your 'agentId'
random.seed(50019733)  # Seed the RNG with the matriculation number of one the group members
MY_AGENT_ID = random.randint(1, 999)
MY_AGENT_ID

66

# Remark: Please put this .ipynb file in the same folder as the provided .zip file to run it properly

# Question 1:

In [ ]:
agent_data = sim_data[sim_data['agentId'] == MY_AGENT_ID]
print("Number of records for this agent:", len(agent_data))
agent_data.head()

# Agent's age
print("Agent Age:", agent_data['age'].unique())

# wake-up and leave-for-work times
schedule = agent_data[['todaysPlan:day', 'todaysPlan:wakeUpTime', 'todaysPlan:leaveTimeForWork']].drop_duplicates()
print("\nWake-up and Leave-for-Work Times:")
print(schedule)

# Find the most visited location
most_visited_location = agent_data['location'].value_counts().idxmax()
visit_reason = agent_data[agent_data['location'] == most_visited_location]['visitReason'].mode()[0]

print("Most visited location (geometry):", most_visited_location)
print("Reason for visiting:", visit_reason)

# get how many times they visited
visit_count = agent_data['location'].value_counts().max()
print("Number of visits:", visit_count)

Number of records for this agent: 1634
Agent Age: [39.08333333]

Wake-up and Leave-for-Work Times:
        todaysPlan:day todaysPlan:wakeUpTime todaysPlan:leaveTimeForWork
66          2019-07-01          07:27:00.000                07:37:00.000
288066      2019-07-02          08:12:00.000                08:22:00.000
576066      2019-07-03          08:13:00.000                08:23:00.000
865066      2019-07-04          07:54:00.000                08:04:00.000
1153066     2019-07-05          07:54:00.000                08:04:00.000
1440066     2019-07-06          07:54:00.000                         NaN
Most visited location (geometry): POINT (7.096120565994761 50.740093514489125)
Reason for visiting: Home_ComingBackFromPub
Number of visits: 558


# Question 2

In [ ]:
# Find the most visited location
most_visited_location = agent_data['location'].value_counts().idxmax()
visit_reason = agent_data[agent_data['location'] == most_visited_location]['visitReason'].mode()[0]

print("Most visited location (geometry):", most_visited_location)
print("Reason for visiting:", visit_reason)

# get how many times they visited
visit_count = agent_data['location'].value_counts().max()
print("Number of visits:", visit_count)

Most visited location (geometry): POINT (7.096120565994761 50.740093514489125)
Reason for visiting: Home_ComingBackFromPub
Number of visits: 558


# Question 3

In [78]:
# Filter data for the specific day
agent_day_data = agent_data[agent_data['todaysPlan:day'] == '2019-07-02']

# Display a quick summary
print("Activity summary on 2019-07-02:")
print(agent_day_data['visitReason'].value_counts())
print("\nTransport modes used:")
print(agent_day_data['currentMode'].value_counts())

# Plot the trajectory
plot_agent_trajectories(agent_day_data)


Activity summary on 2019-07-02:
visitReason
Workplace_Work                 1
Restaurant_WantToEatOutside    1
ComingBackFromRestaurant       1
Home_ComingBackFromWork        1
Name: count, dtype: int64

Transport modes used:
currentMode
AtHome          181
AtWork           99
Transport         4
AtRestaurant      4
Name: count, dtype: int64
